# Week 7: Deep Learning Foundations
## Notebook 13: Learning Rate Schedules: Cosine Annealing with Warmup

### 1. Conceptual Intuition & Theoretical Background
Setting a constant learning rate throughout training often leads to training divergence (if set too high early on) or premature stagnation (if set too low).

**Cosine Annealing Schedule with Linear Warmup:**
1. **Linear Warmup Phase ($t < T_{\text{warmup}}$):**
   Ramps learning rate from $0$ to $\eta_{\max}$ linearly:
   $$\eta_t = \eta_{\max} \cdot \frac{t}{T_{\text{warmup}}}$$
   Prevents gradient explosion in early iterations when weights are random and gradients are noisy.

2. **Cosine Annealing Decay Phase ($t \ge T_{\text{warmup}}$):**
   Smoothly decays learning rate from $\eta_{\max}$ to $\eta_{\min}$ following a half-cosine wave:
   $$\eta_t = \eta_{\min} + \frac{1}{2} (\eta_{\max} - \eta_{\min}) \left( 1 + \cos\left( \pi \cdot \frac{t - T_{\text{warmup}}}{T_{\max} - T_{\text{warmup}}} \right) \right)$$
   Provides smooth, gradient-friendly decay without the abrupt shocks of step-decay schedulers.

---

### 2. Scratch Python Implementation: Cosine Warmup Engine

In [ ]:
import numpy as np

class CosineWarmupSchedulerScratch:
    def __init__(self, lr_max=0.01, lr_min=1e-5, warmup_steps=200, total_steps=1000):
        self.lr_max = lr_max
        self.lr_min = lr_min
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        
    def get_lr(self, current_step):
        if current_step < self.warmup_steps:
            # Linear Warmup
            return self.lr_max * (current_step / max(1, self.warmup_steps))
            
        # Cosine Annealing Decay
        progress = (current_step - self.warmup_steps) / max(1, self.total_steps - self.warmup_steps)
        cosine_decay = 0.5 * (1.0 + np.cos(np.pi * np.clip(progress, 0.0, 1.0)))
        return self.lr_min + (self.lr_max - self.lr_min) * cosine_decay

# Simulate 1,000 Training Steps
scheduler = CosineWarmupSchedulerScratch(lr_max=0.001, lr_min=1e-6, warmup_steps=150, total_steps=1000)

steps = np.arange(1001)
lrs = [scheduler.get_lr(s) for s in steps]

print("--- Cosine Warmup Schedule Trajectory ---")
print(f"Step 0 (Start):        {lrs[0]:.2e}")
print(f"Step 75 (Mid-Warmup):  {lrs[75]:.6f}")
print(f"Step 150 (Warmup Peak): {lrs[150]:.6f} (Reached lr_max)")
print(f"Step 575 (Mid Decay):   {lrs[575]:.6f}")
print(f"Step 1000 (Final):     {lrs[1000]:.2e} (Reached lr_min)")

### 3. Step-by-Step Code Walkthrough
1. **Warmup Ratio:** Linearly increases step size to allow early gradient directions to stabilize.
2. **Cosine Curve:** Decay slows near the end, letting parameters settle into deep local minima.
3. **Boundary Floor ($eta_{min}$):** Maintains a non-zero learning rate to prevent early optimization freeze.

---

### 4. Industry Applications & Trade-offs
- **Foundation LLM Pretraining (LLaMA, GPT-4):** Standard schedule for multi-month cluster pretraining runs.
- **Trade-off:** Requires knowing total pretraining steps $T_{\max}$ in advance.

---

### 5. Top Interview Questions & Answers
**Q1: Why is learning rate warmup necessary when training Transformers with AdamW?**
> **Answer:** Early in training, Adam's second-moment accumulator ($v_t$) is uncalibrated, producing noisy updates. Large early steps destabilize LayerNorm and self-attention weights. Warmup keeps steps small until second-moment estimates stabilize.

**Q2: What is the "1Cycle Policy" and how does it differ from standard cosine annealing?**
> **Answer:** The 1Cycle policy (Leslie Smith) cycles both learning rate and momentum in opposition: as learning rate rises, momentum drops; as learning rate falls, momentum increases. This achieves faster convergence in fewer epochs.